# SDH exp_015 — LGBM 전용 피처 공간 탐색
LGBM 설정은 고정하고 26개 피처 case를 비교합니다. 셀을 위에서부터 하나씩 실행하세요.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'experiments').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'experiments').exists():
    raise RuntimeError('저장소 내부에서 notebook을 실행해 주세요.')
EXP_DIR = PROJECT_ROOT / 'experiments/SDH/exp_015_lgbm_feature_spaces'
sys.path.insert(0, str(EXP_DIR))
import feature_experiment as exp
print('root:', PROJECT_ROOT)
print('cases:', len(exp.CASES))

In [ ]:
train = pd.read_csv(PROJECT_ROOT / 'data/raw/train.csv')
genes = [column for column in train.columns if column not in {'ID', 'SUBCLASS'}]
print(train.shape, len(genes), train['SUBCLASS'].nunique())
pd.DataFrame([{'case': c.name, 'description': c.description} for c in exp.CASES])

## 1. seed 42 fold별 전체 피처를 한 번만 준비

In [ ]:
folds42 = exp.prepare_seed(train, genes, seed=42)
screen_results = {}

## 2A. 표현 방식 7개

In [ ]:
screen_results.update(exp.evaluate_group(folds42, 'representation', seed=42))
exp.leaderboard(screen_results)

## 2B. mutation gene 최소 support 6개

In [ ]:
screen_results.update(exp.evaluate_group(folds42, 'gene_support', seed=42))
exp.leaderboard(screen_results)

## 2C. 개별 블록 ablation과 고정 binning 9개

In [ ]:
screen_results.update(exp.evaluate_group(folds42, 'blocks_and_bins', seed=42))
exp.leaderboard(screen_results)

## 2D. fold-train gain Top-K 4개 (가장 오래 걸림)

In [ ]:
screen_results.update(exp.evaluate_group(folds42, 'gain_topk', seed=42))
screen_table = exp.leaderboard(screen_results)
screen_table

## 3. screen 결과 저장 및 상위 후보 자동 선택

In [ ]:
RESULT_DIR = EXP_DIR / 'results'
RESULT_DIR.mkdir(exist_ok=True)
screen_table.to_csv(RESULT_DIR / 'seed42_screen.csv', index=False)
baseline = float(screen_table.loc[screen_table['case'].eq('F00_full'), 'oof_f1_macro'].iloc[0])
better = screen_table[screen_table['oof_f1_macro'] > baseline]['case'].tolist()
confirmed_names = (better[:3] if better else screen_table.head(3)['case'].tolist())
print('F00 baseline:', baseline)
print('3-seed candidates:', confirmed_names)

## 4. LR 챔피언과 예측 다양성 확인

In [ ]:
lr42 = exp.evaluate_lr_reference(folds42, seed=42)
diversity_table = exp.diversity_vs_lr(screen_results, lr42)
diversity_table.to_csv(RESULT_DIR / 'diversity_vs_lr_seed42.csv', index=False)
print('LR OOF Macro F1:', lr42['oof_f1_macro'])
diversity_table.head(10)

## 5A. LR + 상위 LGBM 확률 앙상블

In [ ]:
top_lgbm_names = screen_table.head(5)['case'].tolist()
lr_blends = exp.search_lr_blends(folds42, screen_results, lr42, top_lgbm_names)
lr_blends.to_csv(RESULT_DIR / 'lr_lgbm_blends_seed42.csv', index=False)
lr_blends.head(20)

## 5B. 서로 다른 LGBM 피처 공간끼리 앙상블

In [ ]:
lgbm_pair_blends = exp.search_lgbm_pair_blends(folds42, screen_results, top_lgbm_names)
lgbm_pair_blends.to_csv(RESULT_DIR / 'lgbm_pair_blends_seed42.csv', index=False)
lgbm_pair_blends.head(20)

## 6. 상위 LR+LGBM 앙상블 3-seed 확인

In [ ]:
ensemble_configs = list(dict.fromkeys(
    (row.case, float(row.lgbm_weight)) for row in lr_blends.head(3).itertuples()
))
print('ensemble configs:', ensemble_configs)
ensemble_confirmation = exp.confirm_lr_blends(train, genes, ensemble_configs, seeds=(42, 52, 62))
ensemble_confirmation.to_csv(RESULT_DIR / 'ensemble_three_seed.csv', index=False)
ensemble_confirmation.groupby(['case', 'lgbm_weight']).agg(
    f1_mean=('oof_f1_macro', 'mean'),
    f1_std=('oof_f1_macro', 'std'),
    delta_mean=('delta_vs_lr', 'mean'),
    delta_min=('delta_vs_lr', 'min'),
).sort_values('f1_mean', ascending=False)

## 7. 순수 LGBM 상위 후보 3-seed 확인
이 셀은 seed 42도 독립적으로 다시 실행합니다. 재현성 확인을 겸합니다.

In [ ]:
confirmation = exp.confirm_cases(train, genes, confirmed_names, seeds=(42, 52, 62))
confirmation.to_csv(RESULT_DIR / 'three_seed_confirmation.csv', index=False)
confirmation.groupby('case').agg(
    f1_mean=('oof_f1_macro', 'mean'),
    f1_std=('oof_f1_macro', 'std'),
    accuracy_mean=('oof_accuracy', 'mean'),
    features=('features', 'mean'),
).sort_values('f1_mean', ascending=False)

## 해석 원칙
단일 seed 최고점만 채택하지 않습니다. F00 대비 3-seed 평균, seed별 방향, fold 변동, 피처 수를 함께 보고 다음 실험 후보를 정합니다.